In [30]:
# import open ai library
!pip install openai


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [31]:
# import necessary libraries
import os 
from dotenv import load_dotenv
from modules import fetch_website_contents
from IPython.display import display, Markdown
from openai import OpenAI

In [32]:
# Load environment variables from .env file
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

# Check if the API key is loaded correctly
if not api_key:
    print("Error: OPENAI_API_KEY not found in environment variables.")
elif not api_key.startswith("sk-proj-"):
    print("an Api key was found, but it doesn't look like it starts with 'sk-proj-'. Please check the .env file for the correct API key.")
elif api_key.strip() != api_key:
    print("An Api key was found, but it looks like it might have space or tab characters. Please check the .env file for formatting issues.")
else:
    print("API key loaded successfully.")

API key loaded successfully.


In [33]:
# Quick call to Frontier model to get started, as a preview
message = "Hello, GPT! This is my first ever message to you! Hi!"

messages = [{"role": "user", "content": message}]

print(messages)

[{'role': 'user', 'content': 'Hello, GPT! This is my first ever message to you! Hi!'}]


In [34]:
openAI = OpenAI()

response = openAI.chat.completions.create(model="gpt-5-nano", messages=messages)
print(response.choices[0].message.content)

Hi there! Welcome, and nice to meet you. I’m here to help with questions, explanations, writing, brainstorming, coding, planning, and more. What would you like to do today?

A few ideas:
- Learn about a topic you’re curious about
- Help with homework or coding
- Write something (email, story, resume, poem)
- Plan a trip or event
- Practice a language or have a casual chat

Tell me one thing you’re interested in, or say “surprise me” and I’ll pick something to start with.


In [35]:
# try out website
# https://www.cnn.com/
cnn = fetch_website_contents("https://www.cnn.com/")
print(cnn)

('Breaking News, Latest News and Videos | CNN', "n\nCNN values your feedback\n1. How relevant is this ad to you?\n2. Did you encounter any technical issues?\nVideo player was slow to load content\nVideo content never loaded\nAd froze or did not finish loading\nVideo content did not start after ad\nAudio on ad was too loud\nOther issues\nAd never loaded\nAd prevented/slowed the page from loading\nContent moved around while ad loaded\nAd was repetitive to ads I've seen previously\nOther issues\nCancel\nSubmit\nThank You!\nYour effort and contribution in providing this feedback is much\n                                        appreciated.\nClose\nAd Feedback\nClose icon\nUS\nWorld\nPolitics\nBusiness\nHealth\nEntertainment\nUnderscored\nStyle\nTravel\nSports\nScience\nClimate\nWeather\nUkraine-Russia War\nIsrael-Hamas War\nGames\nAmazon Prime Day\nMore\nUS\nWorld\nPolitics\nBusiness\nHealth\nEntertainment\nUnderscored\nStyle\nTravel\nSports\nScience\nClimate\nWeather\nUkraine-Russia War\n

In [36]:
# Before continue let review type of prompts
# We are using chat GPT
# Model like GPT have been trained to receive instructions in a particular way
# A "system prompts" that tells them what task they are performing and what tone they should use
# A "user prompt" -- the conversation started that they should reply to

In [37]:
# System prompt example
system_prompt = """
You are a helpful assistant that analyzes the content of a website,
and provides a short summary text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [38]:
# User prompt example
user_prompt_prefix = """
Here are the contests of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.
"""

In [39]:
# Messages
# The API from OpenAI expects to receive messages in a particular structure. Many of those APIs share this structure.

In [40]:
# Example of messages
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What is AI?"}
]

response = openAI.chat.completions.create(model="gpt-4.1-nano", messages=messages)
print(response.choices[0].message.content)

AI, or Artificial Intelligence, refers to the simulation of human intelligence processes by machines, especially computer systems. These processes include learning (the ability to improve performance based on experience), reasoning (the ability to make decisions or solve problems), understanding language, perceiving visual information, and more. 

AI can be categorized into:
- **Narrow AI:** Designed to perform specific tasks (e.g., voice assistants, image recognition).
- **General AI:** Hypothetical future AI that would have the ability to understand, learn, and apply intelligence across a wide range of tasks, similar to human intelligence.

AI is used in various applications, including chatbots, autonomous vehicles, medical diagnosis, recommendation systems, and many others, aiming to improve efficiency and capabilities in numerous fields.


In [41]:
# NOW we can build useful messages for GPT-4.1-mini, using a function
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [44]:
# Try the website
print(messages_for("https://www.cnn.com/"))

[{'role': 'system', 'content': '\nYou are a helpful assistant that analyzes the content of a website,\nand provides a short summary text that might be navigation related.\nRespond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.\n'}, {'role': 'user', 'content': '\nHere are the contests of a website.\nProvide a short summary of this website.\nIf it includes news or announcements, then summarize these too.\nhttps://www.cnn.com/'}]


In [45]:
# Time to bring it all together - The API for OpenAI is pretty simple.
def summarize_website(url):
    website = fetch_website_contents(url)
    response = openAI.chat.completions.create(model="gpt-4.1-mini", messages=messages_for(website))
    return response.choices[0].message.content

In [46]:
summ = summarize_website("https://www.cnn.com/")
print(summ)

TypeError: can only concatenate str (not "tuple") to str